# GNN

In [ ]:
import networkx as nx
import pandas as pd

# Build bipartite graph from account-device relationships
G = nx.Graph()

for _, row in account_device_df.iterrows():
    G.add_edge(f"account_{row.account_id}", 
               f"device_{row.device_id}",
               weight=row.n_sessions,
               last_seen=row.last_seen_ts)

# Degree features
account_nodes = [n for n in G.nodes if n.startswith('account_')]

degree_features = pd.DataFrame({
    'account_id':    [n.replace('account_', '') for n in account_nodes],
    'device_degree': [G.degree(n) for n in account_nodes]
})

# Neighborhood fraud rate
fraud_accounts = set(confirmed_fraud_df['account_id'].astype(str))

def neighborhood_fraud_rate(node, G, fraud_set, hops=1):
    neighbors = set(nx.single_source_shortest_path_length(
        G, node, cutoff=hops).keys()) - {node}
    account_neighbors = {n for n in neighbors if n.startswith('account_')}
    if not account_neighbors:
        return 0.0
    fraud_neighbors = account_neighbors & {f'account_{a}' for a in fraud_set}
    return len(fraud_neighbors) / len(account_neighbors)

degree_features['fraud_rate_1hop'] = [
    neighborhood_fraud_rate(n, G, fraud_accounts, hops=1) 
    for n in account_nodes
]

# PageRank
pagerank = nx.pagerank(G, weight='weight')
degree_features['pagerank'] = [pagerank[n] for n in account_nodes]